# Notebook 00 - Setup dan Eksplorasi Data
## Revisi Pengujian V3 - Skripsi Prediksi Diabetes (Random Forest vs KNN vs SVM)

Notebook ini adalah **fondasi** dari seluruh rangkaian revisi V3. Semua notebook lanjutan
(`01` sampai `06`) memakai preamble, konstanta, pipeline, dan fungsi evaluasi yang **sama persis**
dengan yang didefinisikan di sini, sesuai kontrak teknis pada berkas `_SPEC_BERSAMA.md`.
Dengan begitu setiap angka yang muncul di notebook mana pun berasal dari basis data dan
prosedur yang identik, sehingga antar-eksperimen dapat dibandingkan secara adil.

---

### Latar belakang: empat poin revisi dari penguji

Pada sidang sebelumnya (versi V2), penguji menyampaikan empat catatan utama:

| No | Catatan penguji | Inti masalah |
|:--:|---|---|
| **1** | Tidak ada justifikasi pemilihan **k = 20/21** pada algoritme KNN | Nilai k hanya disebut sebagai "hasil tuning" tanpa bukti eksperimen, kurva, maupun aturan pemilihan yang eksplisit |
| **2** | Tidak ada justifikasi pemilihan **hyperplane** pada SVM | Tidak dijelaskan mengapa kernel linear dipilih, bagaimana parameter C menentukan margin, dan bagaimana bidang pemisah terbentuk |
| **3** | Tidak ada justifikasi rasio pembagian data **80:20** | Rasio 80:20 dipakai begitu saja tanpa membandingkan alternatif rasio lain maupun analisis stabilitas estimasi |
| **4** | **Pengujian kurang banyak** | Evaluasi hanya bersandar pada satu kali holdout split, tanpa validasi silang berulang, uji statistik, uji ketahanan, maupun analisis ablasi |

---

### Peta jawaban: notebook mana menjawab poin apa

| Notebook | Isi utama | Menjawab poin revisi |
|---|---|:--:|
| `00_Setup_dan_Eksplorasi_Data.ipynb` | Preamble bersama, EDA, deteksi outlier, verifikasi reproduksi angka V2 | Fondasi (semua poin) |
| `01_Justifikasi_Rasio_Split.ipynb` | Perbandingan rasio 60:40 s.d. 90:10, learning curve, margin of error, stabilitas estimasi | **Poin 3** |
| `02_Justifikasi_Pemilihan_K_KNN.ipynb` | Sweep nilai k, kurva elbow, aturan one-standard-error, grid `weights` x `metric` | **Poin 1** |
| `03_Justifikasi_Hyperplane_SVM.ipynb` | Perbandingan kernel, grid C/gamma, analisis margin dan support vector, visualisasi hyperplane, bobot w | **Poin 2** |
| `04_Validasi_Statistik_dan_Threshold.ipynb` | Repeated Stratified CV, nested CV, uji statistik antar model, strategi threshold, kurva kalibrasi | **Poin 4** |
| `05_Ablation_Robustness_dan_Keputusan_Model.ipynb` | Ablasi resampling dan fitur, uji ketahanan terhadap noise/missing, analisis subgrup, matriks keputusan | **Poin 4** |
| `06_Model_Final_dan_Export_Produksi.ipynb` | Pelatihan model final, ekspor `rf_model.pkl`, `scaler.pkl`, `model_metadata.json`, `experiments.json` | Sinkronisasi web |

---

### Apa yang dikerjakan notebook 00 ini

1. **CELL 1-6** : preamble wajib (instalasi, import dan konstanta global, utilitas penyimpanan,
   pemuatan dan pembersihan data, pabrik pipeline, fungsi evaluasi standar).
2. **CELL 7-10** : eksplorasi data (distribusi kelas, matriks korelasi, distribusi tiap fitur
   per kelas, deteksi outlier IQR sebelum winsorization).
3. **CELL 11** : **verifikasi reproduksi baseline V2** - memastikan pipeline V3 menghasilkan
   angka yang sama dengan laporan skripsi V2, sehingga seluruh eksperimen tambahan berdiri
   di atas basis yang sah.
4. **CELL 12** : menyimpan data bersih agar dapat dipakai ulang oleh notebook lain.
5. **CELL 13** : ringkasan siap salin untuk naskah skripsi.

> **Catatan menjalankan:** jalankan seluruh cell berurutan dari atas ke bawah. Bila ingin hasil
> tersimpan permanen (dan dibaca oleh notebook `06`), ubah `PAKAI_DRIVE = True` pada CELL 2.

---
## Bagian A - Preamble Wajib (CELL 1-6)

Enam cell berikut adalah **kontrak bersama** seluruh notebook revisi V3. Isinya disalin persis
sama di notebook `01` sampai `06`. Jangan diubah sebagian saja, karena perubahan sekecil apa pun
akan membuat angka antar-notebook tidak lagi sebanding.

In [ ]:
# ============================================================
# CELL 1: Instalasi Library
# ============================================================
!pip install -q pandas numpy matplotlib seaborn scikit-learn imbalanced-learn statsmodels kagglehub

In [ ]:
# ============================================================
# CELL 2: Import & Konstanta Global
# ============================================================
import os, json, time, math, warnings, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, StratifiedShuffleSplit,
    RepeatedStratifiedKFold, cross_validate, cross_val_predict,
    learning_curve, validation_curve, GridSearchCV, RandomizedSearchCV
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, average_precision_score,
    precision_recall_curve, confusion_matrix, classification_report,
    brier_score_loss
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

SELECTED_FEATURES = ['age', 'bmi', 'hypertension', 'HbA1c_level', 'blood_glucose_level']
FEATURE_LABELS    = ['Usia', 'BMI', 'Hipertensi', 'HbA1c', 'Kadar Glukosa']
TARGET            = 'diabetes'

WARNA_MODEL = {'Random Forest': '#3498db', 'KNN': '#e74c3c', 'SVM (Linear)': '#2ecc71'}
WARNA_AKSEN = '#f39c12'

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25
sns.set_style('whitegrid')

# --- Folder output -------------------------------------------------------
# Set PAKAI_DRIVE = True bila ingin hasil tersimpan permanen di Google Drive
# (WAJIB True kalau ingin notebook 06 membaca hasil notebook 01-05).
PAKAI_DRIVE = False

if PAKAI_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/DiaPredict_Revisi'
else:
    OUTPUT_DIR = '/content/hasil_revisi'

for sub in ['', '/tabel', '/gambar', '/json']:
    os.makedirs(OUTPUT_DIR + sub, exist_ok=True)

print(f'Folder output : {OUTPUT_DIR}')
print(f'Fitur         : {SELECTED_FEATURES}')

In [ ]:
# ============================================================
# CELL 3: Fungsi Utilitas Penyimpanan Hasil
# ============================================================
def simpan_tabel(df, nama, tampilkan=True):
    """Simpan DataFrame ke CSV di OUTPUT_DIR/tabel dan tampilkan."""
    path = f'{OUTPUT_DIR}/tabel/{nama}.csv'
    df.to_csv(path, index=False)
    print(f'[TABEL DISIMPAN] {path}')
    if tampilkan:
        display(df)
    return df

def simpan_json(obj, nama):
    """Simpan dict/list hasil eksperimen ke JSON (dipakai notebook 06 & website)."""
    path = f'{OUTPUT_DIR}/json/{nama}.json'
    def _konversi(o):
        if isinstance(o, (np.integer,)):  return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, (np.ndarray,)):  return o.tolist()
        if isinstance(o, (np.bool_,)):    return bool(o)
        return str(o)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=_konversi)
    print(f'[JSON DISIMPAN] {path}')
    return obj

def simpan_gambar(nama, fig=None, dpi=150):
    """Simpan figure matplotlib aktif ke OUTPUT_DIR/gambar."""
    path = f'{OUTPUT_DIR}/gambar/{nama}.png'
    (fig or plt).savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'[GAMBAR DISIMPAN] {path}')
    return path

def garis(judul='', lebar=70):
    print('=' * lebar)
    if judul:
        print(f'  {judul}')
        print('=' * lebar)

In [ ]:
# ============================================================
# CELL 4: Load Dataset + Cleaning + Winsorization
# (Identik dengan pipeline notebook V2 agar hasil dapat dibandingkan)
# ============================================================
import kagglehub

def muat_dan_bersihkan_data(verbose=True):
    path = kagglehub.dataset_download('iammustafatz/diabetes-prediction-dataset')
    csv_file = os.path.join(path, 'diabetes_prediction_dataset.csv')
    df_raw = pd.read_csv(csv_file)

    # 1) Hapus duplikat pada dataset penuh (SAMA seperti V2 -> sisa 96.146 baris)
    df = df_raw.drop_duplicates().reset_index(drop=True)

    # 2) Ambil 5 fitur terpilih + target
    df = df[SELECTED_FEATURES + [TARGET]].copy()

    # 3) Winsorization (capping IQR) hanya untuk fitur numerik non-biner
    numeric_feats = [f for f in SELECTED_FEATURES if df[f].nunique() > 2]
    ringkas = []
    for feat in numeric_feats:
        Q1, Q3 = df[feat].quantile(0.25), df[feat].quantile(0.75)
        IQR = Q3 - Q1
        low, up = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        n_cap = int(((df[feat] < low) | (df[feat] > up)).sum())
        df[feat] = df[feat].clip(lower=low, upper=up)
        ringkas.append({'fitur': feat, 'batas_bawah': low, 'batas_atas': up, 'n_dicapping': n_cap})

    if verbose:
        garis('DATA SIAP PAKAI')
        print(f'Baris (setelah hapus duplikat) : {len(df):,}')
        print(f'Distribusi kelas               : '
              f'{(df[TARGET]==0).sum():,} sehat / {(df[TARGET]==1).sum():,} diabetes '
              f'({df[TARGET].mean()*100:.2f}% positif)')
        display(pd.DataFrame(ringkas))
    return df

df_clean = muat_dan_bersihkan_data()
X_all = df_clean[SELECTED_FEATURES].copy()
y_all = df_clean[TARGET].copy()

In [ ]:
# ============================================================
# CELL 5: Pabrik Pipeline Model (anti data leakage)
# Urutan: StandardScaler -> SMOTE -> Classifier (imblearn Pipeline,
# sehingga SMOTE HANYA aktif saat fit, tidak saat predict/validasi)
# ============================================================

# Hyperparameter terbaik hasil tuning notebook V2 (baseline pembanding)
PARAM_RF_V2  = dict(n_estimators=200, max_depth=10, min_samples_split=5,
                    min_samples_leaf=4, max_features='log2', criterion='entropy',
                    class_weight='balanced')
PARAM_KNN_V2 = dict(n_neighbors=21, weights='uniform', metric='euclidean', leaf_size=20)
PARAM_SVM_V2 = dict(C=0.1, max_iter=3000)

def buat_pipeline_rf(pakai_smote=True, **params):
    p = {**PARAM_RF_V2, **params}
    langkah = [('scaler', StandardScaler())]
    if pakai_smote:
        langkah.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    langkah.append(('clf', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, **p)))
    return ImbPipeline(langkah)

def buat_pipeline_knn(pakai_smote=True, **params):
    p = {**PARAM_KNN_V2, **params}
    langkah = [('scaler', StandardScaler())]
    if pakai_smote:
        langkah.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    langkah.append(('clf', KNeighborsClassifier(n_jobs=-1, **p)))
    return ImbPipeline(langkah)

def buat_pipeline_svm(pakai_smote=True, kernel='linear', C=0.1, gamma='scale',
                      degree=3, max_iter=3000, kalibrasi='sigmoid'):
    """kernel='linear' -> LinearSVC (cepat). Kernel lain -> SVC."""
    if kernel == 'linear':
        base = LinearSVC(C=C, max_iter=max_iter, class_weight='balanced',
                         dual=False, random_state=RANDOM_STATE)
    else:
        base = SVC(kernel=kernel, C=C, gamma=gamma, degree=degree,
                   class_weight='balanced', random_state=RANDOM_STATE)
    langkah = [('scaler', StandardScaler())]
    if pakai_smote:
        langkah.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    langkah.append(('clf', CalibratedClassifierCV(base, cv=3, method=kalibrasi)))
    return ImbPipeline(langkah)

PABRIK_MODEL = {
    'Random Forest': buat_pipeline_rf,
    'KNN'          : buat_pipeline_knn,
    'SVM (Linear)' : buat_pipeline_svm,
}

In [ ]:
# ============================================================
# CELL 6: Fungsi Evaluasi Standar (dipakai seluruh notebook)
# ============================================================
def threshold_youden(y_true, y_proba):
    fpr, tpr, thr = roc_curve(y_true, y_proba)
    return float(thr[np.argmax(tpr - fpr)])

def hitung_metrik(y_true, y_pred, y_proba=None):
    hasil = {
        'accuracy' : accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall'   : recall_score(y_true, y_pred, zero_division=0),
        'f1'       : f1_score(y_true, y_pred, zero_division=0),
    }
    if y_proba is not None:
        hasil['roc_auc']  = roc_auc_score(y_true, y_proba)
        hasil['ap_score'] = average_precision_score(y_true, y_proba)
        hasil['brier']    = brier_score_loss(y_true, y_proba)
    return hasil

def evaluasi_holdout(model, X_tr, y_tr, X_te, y_te, tuning_threshold=True):
    """Fit -> prediksi -> metrik pada threshold 0.5 dan threshold Youden."""
    t0 = time.time(); model.fit(X_tr, y_tr); waktu_latih = time.time() - t0
    t0 = time.time(); y_proba = model.predict_proba(X_te)[:, 1]; waktu_infer = time.time() - t0

    thr = threshold_youden(y_te, y_proba) if tuning_threshold else 0.5
    m_def  = hitung_metrik(y_te, (y_proba >= 0.5).astype(int), y_proba)
    m_tune = hitung_metrik(y_te, (y_proba >= thr).astype(int), y_proba)
    return {
        'threshold': thr,
        'waktu_latih_s': waktu_latih,
        'waktu_infer_ms': waktu_infer * 1000,
        **{f'{k}_default': v for k, v in m_def.items()},
        **{f'{k}_tuned'  : v for k, v in m_tune.items()},
    }

def ci95_proporsi(p, n):
    """Confidence interval 95% (Wald) untuk metrik berbasis proporsi (mis. recall)."""
    if n == 0: return (np.nan, np.nan, np.nan)
    se = math.sqrt(max(p * (1 - p), 1e-12) / n)
    return (p - 1.96 * se, p + 1.96 * se, 1.96 * se)

---
## Bagian B - Eksplorasi Data (CELL 7-10)

Bagian ini menggambarkan karakteristik data yang menjadi dasar seluruh eksperimen revisi.
Empat hal yang diperiksa: (1) seberapa timpang distribusi kelas, (2) seberapa kuat hubungan
antar fitur dan terhadap target, (3) bagaimana sebaran tiap fitur berbeda antara kelompok
sehat dan diabetes, serta (4) seberapa banyak outlier yang ditangani oleh winsorization.

### CELL 7 - Distribusi Kelas

Ketimpangan kelas adalah alasan utama mengapa **akurasi saja tidak cukup** sebagai tolok ukur.
Pada data ini kelas positif (diabetes) hanya sekitar 8,5 persen, sehingga model yang menebak
"semua sehat" pun sudah memperoleh akurasi di atas 91 persen tanpa berguna sama sekali.
Karena itu seluruh notebook revisi menekankan **recall**, **F1**, dan **ROC-AUC**.

In [ ]:
# ============================================================
# CELL 7: EDA - Distribusi Kelas Target
# ============================================================
garis('EDA 1: DISTRIBUSI KELAS TARGET')

n_total   = len(df_clean)
n_sehat   = int((df_clean[TARGET] == 0).sum())
n_diabet  = int((df_clean[TARGET] == 1).sum())
pct_sehat = n_sehat / n_total * 100
pct_diab  = n_diabet / n_total * 100
rasio_imb = n_sehat / max(n_diabet, 1)

print(f'Total baris          : {n_total:,}')
print(f'Kelas 0 (sehat)      : {n_sehat:,} ({pct_sehat:.2f}%)')
print(f'Kelas 1 (diabetes)   : {n_diabet:,} ({pct_diab:.2f}%)')
print(f'Rasio ketimpangan    : 1 : {rasio_imb:.2f} (positif : negatif)')
print(f'Akurasi "tebak semua sehat" : {pct_sehat:.2f}% -> baseline naif yang menyesatkan')

df_kelas = pd.DataFrame([
    {'kelas': 0, 'label': 'Sehat (Non-Diabetes)', 'jumlah': n_sehat,  'persen': round(pct_sehat, 4)},
    {'kelas': 1, 'label': 'Diabetes',             'jumlah': n_diabet, 'persen': round(pct_diab, 4)},
])
simpan_tabel(df_kelas, 'eda_distribusi_kelas')

fig, ax = plt.subplots(1, 2, figsize=(13, 5))

warna_kelas = ['#3498db', '#e74c3c']
bar = ax[0].bar(df_kelas['label'], df_kelas['jumlah'], color=warna_kelas,
                edgecolor='black', linewidth=0.8, width=0.6)
for b, j, p in zip(bar, df_kelas['jumlah'], df_kelas['persen']):
    ax[0].text(b.get_x() + b.get_width() / 2, b.get_height() + n_total * 0.012,
               f'{j:,}\n({p:.2f}%)', ha='center', va='bottom', fontweight='bold')
ax[0].set_ylabel('Jumlah Sampel')
ax[0].set_title('Distribusi Kelas Target (Jumlah Absolut)', fontweight='bold')
ax[0].set_ylim(0, n_total * 1.08)

ax[1].pie(df_kelas['jumlah'], labels=df_kelas['label'], colors=warna_kelas,
          autopct='%1.2f%%', startangle=90, explode=(0, 0.08),
          wedgeprops={'edgecolor': 'black', 'linewidth': 0.8},
          textprops={'fontsize': 11})
ax[1].set_title('Proporsi Kelas Target', fontweight='bold')
ax[1].grid(False)

plt.suptitle('EDA - Ketimpangan Kelas pada Dataset Prediksi Diabetes',
             fontsize=14, fontweight='bold')
plt.tight_layout()
simpan_gambar('eda_distribusi_kelas')
plt.show()

garis('KESIMPULAN EDA 1')
print(f'Dataset sangat tidak seimbang: hanya {pct_diab:.2f}% kasus positif dari {n_total:,} baris.')
print('Implikasi metodologis:')
print('  1. Akurasi tidak dipakai sebagai metrik utama (baseline naif sudah 91,5%).')
print('  2. SMOTE diterapkan HANYA pada data latih di dalam imblearn Pipeline.')
print('  3. Semua pembagian data dan validasi silang bersifat STRATIFIED.')
print('  4. Metrik utama: Recall (menekan false negative), F1, dan ROC-AUC.')

### CELL 8 - Matriks Korelasi

Matriks korelasi Pearson dipakai untuk dua hal: memeriksa **multikolinearitas** antar fitur
(korelasi antar prediktor yang terlalu tinggi akan membuat bobot model tidak stabil, terutama
pada SVM linear) dan melihat **kekuatan hubungan tiap fitur terhadap target**.

In [ ]:
# ============================================================
# CELL 8: EDA - Matriks Korelasi 5 Fitur + Target
# ============================================================
garis('EDA 2: MATRIKS KORELASI (5 FITUR + TARGET)')

kolom_analisis = SELECTED_FEATURES + [TARGET]
label_analisis = FEATURE_LABELS + ['Diabetes']

korelasi = df_clean[kolom_analisis].corr(method='pearson')
korelasi_label = korelasi.copy()
korelasi_label.index = label_analisis
korelasi_label.columns = label_analisis

df_korelasi = korelasi_label.round(4).reset_index().rename(columns={'index': 'variabel'})
simpan_tabel(df_korelasi, 'eda_korelasi')

fig, ax = plt.subplots(1, 2, figsize=(15, 6))

mask = np.triu(np.ones_like(korelasi_label, dtype=bool), k=1)
sns.heatmap(korelasi_label, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.8,
            cbar_kws={'label': 'Koefisien Korelasi Pearson'}, ax=ax[0])
ax[0].set_title('Matriks Korelasi Antar Variabel', fontweight='bold')
ax[0].grid(False)

kor_target = korelasi[TARGET].drop(TARGET)
urut = kor_target.sort_values(ascending=True)
label_urut = [FEATURE_LABELS[SELECTED_FEATURES.index(f)] for f in urut.index]
warna_bar = [WARNA_AKSEN if v == urut.max() else '#3498db' for v in urut.values]
ax[1].barh(label_urut, urut.values, color=warna_bar, edgecolor='black', linewidth=0.7)
for i, v in enumerate(urut.values):
    ax[1].text(v + 0.008, i, f'{v:.3f}', va='center', fontweight='bold')
ax[1].set_xlabel('Korelasi terhadap Target (Diabetes)')
ax[1].set_title('Kekuatan Hubungan Tiap Fitur dengan Target', fontweight='bold')
ax[1].set_xlim(0, max(urut.values) * 1.25)

plt.suptitle('EDA - Analisis Korelasi Fitur', fontsize=14, fontweight='bold')
plt.tight_layout()
simpan_gambar('eda_korelasi')
plt.show()

# Cek multikolinearitas antar fitur (di luar target)
kor_fitur = korelasi.loc[SELECTED_FEATURES, SELECTED_FEATURES].abs().to_numpy(copy=True)
np.fill_diagonal(kor_fitur, 0.0)   # abaikan diagonal (korelasi fitur dengan dirinya = 1)
maks_antar_fitur = float(kor_fitur.max())
pos = np.unravel_index(int(np.argmax(kor_fitur)), kor_fitur.shape)
pasangan = (SELECTED_FEATURES[pos[0]], SELECTED_FEATURES[pos[1]])

garis('KESIMPULAN EDA 2')
print('Korelasi tiap fitur terhadap target (urut dari terkuat):')
for f, v in kor_target.sort_values(ascending=False).items():
    nama = FEATURE_LABELS[SELECTED_FEATURES.index(f)]
    print(f'  {nama:<15s} : {v:+.4f}')
print()
print(f'Korelasi absolut tertinggi ANTAR FITUR : {maks_antar_fitur:.4f} '
      f'(pasangan {pasangan[0]} - {pasangan[1]})')
if maks_antar_fitur < 0.7:
    print('Kesimpulan: tidak ada indikasi multikolinearitas berat (semua < 0,70),')
    print('sehingga kelima fitur layak dipertahankan bersama dan bobot SVM linear')
    print('dapat ditafsirkan sebagai kontribusi masing-masing fitur.')
else:
    print('Peringatan: terdapat pasangan fitur dengan korelasi tinggi, perlu ditinjau.')

### CELL 9 - Distribusi Tiap Fitur per Kelas

Bagian ini memeriksa apakah tiap fitur benar-benar memisahkan kelompok sehat dan diabetes.
Fitur numerik ditampilkan dengan **violin plot** (menggabungkan bentuk sebaran dan ringkasan
kuartil), sedangkan fitur biner `hypertension` ditampilkan sebagai **persentase penderita
diabetes** pada masing-masing kelompok.

In [ ]:
# ============================================================
# CELL 9: EDA - Distribusi Tiap Fitur per Kelas
# ============================================================
garis('EDA 3: DISTRIBUSI TIAP FITUR PER KELAS')

fitur_numerik = [f for f in SELECTED_FEATURES if df_clean[f].nunique() > 2]
fitur_biner   = [f for f in SELECTED_FEATURES if df_clean[f].nunique() <= 2]

# --- Tabel ringkasan statistik per kelas ---------------------------------
baris_ringkas = []
for f in SELECTED_FEATURES:
    nama = FEATURE_LABELS[SELECTED_FEATURES.index(f)]
    g0 = df_clean.loc[df_clean[TARGET] == 0, f]
    g1 = df_clean.loc[df_clean[TARGET] == 1, f]
    sd_gab = math.sqrt(((len(g0) - 1) * g0.var() + (len(g1) - 1) * g1.var()) /
                       max(len(g0) + len(g1) - 2, 1))
    cohen_d = (g1.mean() - g0.mean()) / sd_gab if sd_gab > 0 else 0.0
    baris_ringkas.append({
        'fitur'          : f,
        'label'          : nama,
        'tipe'           : 'numerik' if f in fitur_numerik else 'biner',
        'mean_sehat'     : round(float(g0.mean()), 4),
        'mean_diabetes'  : round(float(g1.mean()), 4),
        'median_sehat'   : round(float(g0.median()), 4),
        'median_diabetes': round(float(g1.median()), 4),
        'std_sehat'      : round(float(g0.std()), 4),
        'std_diabetes'   : round(float(g1.std()), 4),
        'selisih_mean'   : round(float(g1.mean() - g0.mean()), 4),
        'cohen_d'        : round(float(cohen_d), 4),
    })

df_dist_fitur = pd.DataFrame(baris_ringkas).sort_values('cohen_d', ascending=False)
simpan_tabel(df_dist_fitur, 'eda_distribusi_fitur')

# --- Visualisasi ---------------------------------------------------------
n_panel = len(SELECTED_FEATURES)
n_kol   = 3
n_baris = math.ceil(n_panel / n_kol)
fig, axes = plt.subplots(n_baris, n_kol, figsize=(16, 4.6 * n_baris))
axes = np.array(axes).reshape(-1)

palet_kelas = {0: '#3498db', 1: '#e74c3c'}
df_plot = df_clean.copy()
df_plot['Kelompok'] = df_plot[TARGET].map({0: 'Sehat', 1: 'Diabetes'})

idx = 0
for f in SELECTED_FEATURES:
    ax = axes[idx]
    nama = FEATURE_LABELS[SELECTED_FEATURES.index(f)]
    if f in fitur_numerik:
        sns.violinplot(data=df_plot, x='Kelompok', y=f, hue='Kelompok',
                       palette=[palet_kelas[0], palet_kelas[1]], legend=False,
                       inner='quartile', cut=0, ax=ax)
        m0 = df_plot.loc[df_plot[TARGET] == 0, f].mean()
        m1 = df_plot.loc[df_plot[TARGET] == 1, f].mean()
        ax.scatter([0, 1], [m0, m1], color=WARNA_AKSEN, s=70, zorder=5,
                   edgecolor='black', linewidth=0.8, label='Rata-rata')
        ax.legend(loc='upper left', fontsize=9)
        ax.set_title(f'Distribusi {nama} per Kelompok', fontweight='bold')
        ax.set_ylabel(nama)
    else:
        # Fitur biner: persentase penderita diabetes per nilai fitur
        ringkas_bin = (df_plot.groupby(f)[TARGET]
                       .agg(['mean', 'count']).reset_index())
        ringkas_bin['persen_diabetes'] = ringkas_bin['mean'] * 100
        label_x = ['Tidak Hipertensi', 'Hipertensi'] if f == 'hypertension' \
                  else [str(v) for v in ringkas_bin[f]]
        bar = ax.bar(label_x, ringkas_bin['persen_diabetes'],
                     color=['#3498db', WARNA_AKSEN], edgecolor='black', linewidth=0.8,
                     width=0.55)
        for b, p, c in zip(bar, ringkas_bin['persen_diabetes'], ringkas_bin['count']):
            ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.4,
                    f'{p:.2f}%\n(n={c:,})', ha='center', va='bottom',
                    fontweight='bold', fontsize=9)
        ax.set_ylabel('Persentase Diabetes (%)')
        ax.set_ylim(0, max(ringkas_bin['persen_diabetes']) * 1.35)
        ax.set_title(f'Prevalensi Diabetes menurut {nama}', fontweight='bold')
        ax.set_xlabel('')
    idx += 1

for j in range(idx, len(axes)):
    axes[j].axis('off')

plt.suptitle('EDA - Sebaran Tiap Fitur pada Kelompok Sehat vs Diabetes',
             fontsize=14, fontweight='bold')
plt.tight_layout()
simpan_gambar('eda_distribusi_fitur')
plt.show()

garis('KESIMPULAN EDA 3')
print('Daya pisah tiap fitur diukur dengan Cohen d (semakin besar, semakin memisahkan):')
for _, r in df_dist_fitur.iterrows():
    d = abs(r['cohen_d'])
    if d >= 0.8:   kat = 'besar'
    elif d >= 0.5: kat = 'sedang'
    elif d >= 0.2: kat = 'kecil'
    else:          kat = 'sangat kecil'
    print(f"  {r['label']:<15s} d = {r['cohen_d']:+.3f}  (efek {kat})")
print()
print('Fitur dengan daya pisah terkuat konsisten dengan literatur klinis: kadar HbA1c dan')
print('glukosa darah adalah penanda diagnostik utama, sedangkan usia, BMI, dan hipertensi')
print('berperan sebagai faktor risiko pendukung.')

### CELL 10 - Deteksi Outlier IQR (Sebelum Winsorization)

Notebook V2 menerapkan **winsorization** (capping IQR pada batas `Q1 - 1,5 x IQR` dan
`Q3 + 1,5 x IQR`) tanpa melaporkan berapa banyak nilai yang terpengaruh. Cell ini memuat ulang
data **mentah** (tanpa capping) untuk menghitung ulang jumlah outlier, lalu membandingkannya
dengan kondisi setelah winsorization. Tujuannya: menunjukkan bahwa penanganan outlier bersifat
**terukur dan proporsional**, bukan pemotongan data sembarangan.

Catatan penting: winsorization **tidak membuang baris**, hanya menggeser nilai ekstrem ke batas
IQR, sehingga jumlah sampel tetap 96.146 baris.

In [ ]:
# ============================================================
# CELL 10: EDA - Deteksi Outlier IQR Sebelum Winsorization
# ============================================================
garis('EDA 4: DETEKSI OUTLIER IQR (SEBELUM WINSORIZATION)')

def muat_data_mentah():
    """Muat data TANPA capping IQR, untuk pembanding analisis outlier.
    Langkah identik dengan muat_dan_bersihkan_data(), tetapi winsorization dilewati."""
    path = kagglehub.dataset_download('iammustafatz/diabetes-prediction-dataset')
    csv_file = os.path.join(path, 'diabetes_prediction_dataset.csv')
    df_raw = pd.read_csv(csv_file)
    df_m = df_raw.drop_duplicates().reset_index(drop=True)
    df_m = df_m[SELECTED_FEATURES + [TARGET]].copy()
    return df_m

df_mentah = muat_data_mentah()
print(f'Baris data mentah (setelah hapus duplikat) : {len(df_mentah):,}')
print(f'Baris data bersih (setelah winsorization)  : {len(df_clean):,}')
print('Jumlah baris identik -> winsorization hanya melakukan capping nilai, bukan penghapusan.')
print()

fitur_num = [f for f in SELECTED_FEATURES if df_mentah[f].nunique() > 2]

baris_outlier = []
batas_iqr = {}
for f in fitur_num:
    s = df_mentah[f]
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    low, up = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    batas_iqr[f] = (low, up)

    n_bawah = int((s < low).sum())
    n_atas  = int((s > up).sum())
    n_out   = n_bawah + n_atas
    n_out_sesudah = int(((df_clean[f] < low) | (df_clean[f] > up)).sum())

    baris_outlier.append({
        'fitur'             : f,
        'label'             : FEATURE_LABELS[SELECTED_FEATURES.index(f)],
        'min_sebelum'       : round(float(s.min()), 4),
        'Q1'                : round(float(Q1), 4),
        'median'            : round(float(s.median()), 4),
        'Q3'                : round(float(Q3), 4),
        'max_sebelum'       : round(float(s.max()), 4),
        'IQR'               : round(float(IQR), 4),
        'batas_bawah'       : round(float(low), 4),
        'batas_atas'        : round(float(up), 4),
        'n_outlier_bawah'   : n_bawah,
        'n_outlier_atas'    : n_atas,
        'n_outlier_total'   : n_out,
        'persen_outlier'    : round(n_out / len(df_mentah) * 100, 4),
        'min_sesudah'       : round(float(df_clean[f].min()), 4),
        'max_sesudah'       : round(float(df_clean[f].max()), 4),
        'n_outlier_sesudah' : n_out_sesudah,
    })

df_outlier = pd.DataFrame(baris_outlier)
simpan_tabel(df_outlier, 'eda_outlier')

# --- Visualisasi boxplot sebelum vs sesudah ------------------------------
n_f = len(fitur_num)
fig, axes = plt.subplots(2, n_f, figsize=(4.0 * n_f, 9))
axes = np.array(axes).reshape(2, n_f)

for j, f in enumerate(fitur_num):
    nama = FEATURE_LABELS[SELECTED_FEATURES.index(f)]
    low, up = batas_iqr[f]

    axes[0, j].boxplot(df_mentah[f].values, vert=True, patch_artist=True, widths=0.5,
                       boxprops={'facecolor': '#e74c3c', 'alpha': 0.65},
                       medianprops={'color': 'black', 'linewidth': 1.6},
                       flierprops={'marker': 'o', 'markersize': 2.5,
                                   'markerfacecolor': '#e74c3c', 'alpha': 0.35})
    axes[0, j].axhline(up, color=WARNA_AKSEN, linestyle='--', linewidth=1.3)
    axes[0, j].axhline(low, color=WARNA_AKSEN, linestyle='--', linewidth=1.3)
    n_out = int(df_outlier.loc[df_outlier['fitur'] == f, 'n_outlier_total'].iloc[0])
    pct_o = float(df_outlier.loc[df_outlier['fitur'] == f, 'persen_outlier'].iloc[0])
    axes[0, j].set_title(f'{nama} - SEBELUM\n{n_out:,} outlier ({pct_o:.2f}%)',
                         fontweight='bold', fontsize=11)
    axes[0, j].set_xticks([])

    axes[1, j].boxplot(df_clean[f].values, vert=True, patch_artist=True, widths=0.5,
                       boxprops={'facecolor': '#2ecc71', 'alpha': 0.65},
                       medianprops={'color': 'black', 'linewidth': 1.6},
                       flierprops={'marker': 'o', 'markersize': 2.5,
                                   'markerfacecolor': '#2ecc71', 'alpha': 0.35})
    axes[1, j].axhline(up, color=WARNA_AKSEN, linestyle='--', linewidth=1.3)
    axes[1, j].axhline(low, color=WARNA_AKSEN, linestyle='--', linewidth=1.3)
    axes[1, j].set_title(f'{nama} - SESUDAH\ncapping ke [{low:.2f}, {up:.2f}]',
                         fontweight='bold', fontsize=11)
    axes[1, j].set_xticks([])

plt.suptitle('EDA - Outlier IQR Sebelum vs Sesudah Winsorization',
             fontsize=14, fontweight='bold')
plt.tight_layout()
simpan_gambar('eda_outlier')
plt.show()

total_out = int(df_outlier['n_outlier_total'].sum())
garis('KESIMPULAN EDA 4')
print(f'Total nilai outlier yang di-capping : {total_out:,} nilai '
      f'({total_out / (len(df_mentah) * len(fitur_num)) * 100:.2f}% dari seluruh sel numerik)')
for _, r in df_outlier.iterrows():
    print(f"  {r['label']:<15s} : {r['n_outlier_total']:>6,} nilai "
          f"({r['persen_outlier']:.2f}%) -> dibatasi ke "
          f"[{r['batas_bawah']:.2f}, {r['batas_atas']:.2f}]")
print()
print('Catatan metodologis:')
print('  1. Winsorization dipilih daripada penghapusan baris agar tidak ada informasi')
print('     pasien yang hilang, terutama pada kelas minoritas yang sudah langka.')
print('  2. Setelah capping, kolom numerik tidak lagi memuat nilai di luar batas IQR')
print('     (kolom n_outlier_sesudah bernilai 0).')
print('  3. Prosedur ini identik dengan notebook V2, sehingga angka baseline tetap sebanding.')

---
## Bagian C - Verifikasi Reproduksi Baseline V2 (CELL 11)

Sebelum menambahkan eksperimen baru, wajib dibuktikan bahwa pipeline V3 di notebook ini
**menghasilkan angka yang sama** dengan yang dilaporkan pada skripsi V2. Bila reproduksi
berhasil, maka semua eksperimen tambahan pada notebook `01`-`05` dapat dibandingkan langsung
dengan hasil lama tanpa perlu menghitung ulang seluruh naskah.

**Prosedur:** split 80:20 stratified dengan `random_state = 42`, tiga pipeline dilatih memakai
hyperparameter hasil tuning V2 (`PARAM_RF_V2`, `PARAM_KNN_V2`, `PARAM_SVM_V2`), lalu dievaluasi
dengan `evaluasi_holdout`. Metrik yang dibandingkan adalah metrik pada **threshold Youden**
(threshold optimal), persis seperti pelaporan V2.

In [ ]:
# ============================================================
# CELL 11: Verifikasi Reproduksi Baseline V2 (Split 80:20, seed 42)
# ============================================================
garis('VERIFIKASI REPRODUKSI BASELINE V2')

# --- Angka referensi dari laporan skripsi V2 (threshold Youden) ----------
REFERENSI_V2 = {
    'Random Forest': {'accuracy': 0.8933, 'precision': 0.4481, 'recall': 0.9057,
                      'f1': 0.5995, 'roc_auc': 0.9733, 'threshold': 0.4965},
    'KNN'          : {'accuracy': 0.8559, 'precision': 0.3710, 'recall': 0.9121,
                      'f1': 0.5274, 'roc_auc': 0.9524, 'threshold': 0.3810},
    'SVM (Linear)' : {'accuracy': 0.8775, 'precision': 0.4097, 'recall': 0.8833,
                      'f1': 0.5598, 'roc_auc': 0.9581, 'threshold': 0.4951},
}
TOLERANSI = 0.01   # selisih absolut <= 0,01 dianggap tereproduksi

# --- Split 80:20 stratified, seed 42 (identik V2) ------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=RANDOM_STATE
)
print(f'Data latih : {len(X_train):,} baris ({y_train.mean()*100:.2f}% positif)')
print(f'Data uji   : {len(X_test):,} baris ({y_test.mean()*100:.2f}% positif)')
print(f'Rasio      : {len(X_train)/len(X_all)*100:.1f}% : {len(X_test)/len(X_all)*100:.1f}%')
print()
print('Estimasi waktu total pelatihan tiga model: sekitar 2-6 menit pada Colab CPU.')
print()

# --- Latih & evaluasi tiga model ----------------------------------------
hasil_model = {}
proba_model = {}
for nama, pabrik in PABRIK_MODEL.items():
    print(f'[PROSES] Melatih {nama} ...')
    t_mulai = time.time()
    model = pabrik()
    hasil = evaluasi_holdout(model, X_train, y_train, X_test, y_test, tuning_threshold=True)
    proba_model[nama] = model.predict_proba(X_test)[:, 1]
    hasil_model[nama] = hasil
    print(f'[SELESAI] {nama} dalam {time.time() - t_mulai:.1f} detik | '
          f'thr={hasil["threshold"]:.4f} | recall={hasil["recall_tuned"]:.4f} | '
          f'f1={hasil["f1_tuned"]:.4f} | auc={hasil["roc_auc_tuned"]:.4f}')
print()

# --- Tabel perbandingan V2 vs sekarang ----------------------------------
PETA_METRIK = [('accuracy', 'Accuracy'), ('precision', 'Precision'),
               ('recall', 'Recall'), ('f1', 'F1-Score'),
               ('roc_auc', 'ROC-AUC'), ('threshold', 'Threshold')]

baris_verif = []
for nama in PABRIK_MODEL.keys():
    h = hasil_model[nama]
    ref = REFERENSI_V2[nama]
    for kunci, label in PETA_METRIK:
        nilai_now = h['threshold'] if kunci == 'threshold' else h[f'{kunci}_tuned']
        nilai_v2  = ref[kunci]
        selisih   = float(nilai_now) - float(nilai_v2)
        baris_verif.append({
            'model'         : nama,
            'metrik'        : label,
            'nilai_v2'      : round(float(nilai_v2), 4),
            'nilai_v3'      : round(float(nilai_now), 4),
            'selisih'       : round(selisih, 4),
            'selisih_abs'   : round(abs(selisih), 4),
            'selisih_persen': round(selisih / max(abs(float(nilai_v2)), 1e-9) * 100, 2),
            'status'        : 'COCOK' if abs(selisih) <= TOLERANSI else 'PERIKSA',
        })

df_verifikasi = pd.DataFrame(baris_verif)
simpan_tabel(df_verifikasi, 'verifikasi_reproduksi_v2')

n_cocok = int((df_verifikasi['status'] == 'COCOK').sum())
n_total_cek = len(df_verifikasi)
selisih_maks = float(df_verifikasi['selisih_abs'].max())
selisih_rata = float(df_verifikasi['selisih_abs'].mean())

# --- Visualisasi ---------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

metrik_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x = np.arange(len(metrik_plot))
lebar = 0.13
for i, nama in enumerate(PABRIK_MODEL.keys()):
    sub = df_verifikasi[df_verifikasi['model'] == nama].set_index('metrik')
    v2  = [sub.loc[m, 'nilai_v2'] for m in metrik_plot]
    v3  = [sub.loc[m, 'nilai_v3'] for m in metrik_plot]
    axes[0].bar(x + (i * 2 - 2.5) * lebar, v2, lebar, label=f'{nama} - V2',
                color=WARNA_MODEL[nama], alpha=0.45, edgecolor='black', linewidth=0.6)
    axes[0].bar(x + (i * 2 - 1.5) * lebar, v3, lebar, label=f'{nama} - V3',
                color=WARNA_MODEL[nama], edgecolor='black', linewidth=0.6)
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrik_plot)
axes[0].set_ylabel('Nilai Metrik')
axes[0].set_ylim(0, 1.12)
axes[0].set_title('Perbandingan Metrik: Laporan V2 vs Reproduksi V3\n(threshold Youden)',
                  fontweight='bold')
axes[0].legend(fontsize=8, ncol=3, loc='upper center')

for nama in PABRIK_MODEL.keys():
    fpr, tpr, _ = roc_curve(y_test, proba_model[nama])
    auc_now = hasil_model[nama]['roc_auc_tuned']
    axes[1].plot(fpr, tpr, color=WARNA_MODEL[nama], linewidth=2.2,
                 label=f'{nama} (AUC = {auc_now:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1.1, label='Tebakan acak (AUC = 0,5)')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Kurva ROC Baseline V3 pada Data Uji (20%)', fontweight='bold')
axes[1].legend(loc='lower right', fontsize=10)

plt.suptitle('Verifikasi Reproduksi Baseline V2 - Split 80:20 Stratified, seed 42',
             fontsize=14, fontweight='bold')
plt.tight_layout()
simpan_gambar('verifikasi_baseline_v2')
plt.show()

# --- Simpan JSON baseline -----------------------------------------------
hasil_baseline_v2 = {
    'deskripsi'      : 'Verifikasi reproduksi angka baseline skripsi V2 memakai pipeline V3',
    'konfigurasi'    : {
        'rasio_split'   : '80:20',
        'stratified'    : True,
        'random_state'  : RANDOM_STATE,
        'n_total'       : int(len(X_all)),
        'n_train'       : int(len(X_train)),
        'n_test'        : int(len(X_test)),
        'fitur'         : SELECTED_FEATURES,
        'pakai_smote'   : True,
        'threshold'     : 'Youden J (tuned)',
        'param_rf'      : PARAM_RF_V2,
        'param_knn'     : PARAM_KNN_V2,
        'param_svm'     : PARAM_SVM_V2,
    },
    'referensi_v2'   : REFERENSI_V2,
    'hasil_v3'       : hasil_model,
    'tabel_verifikasi': df_verifikasi.to_dict('records'),
    'ringkasan'      : {
        'toleransi'         : TOLERANSI,
        'jumlah_dicek'      : n_total_cek,
        'jumlah_cocok'      : n_cocok,
        'selisih_abs_maks'  : round(selisih_maks, 4),
        'selisih_abs_rata'  : round(selisih_rata, 4),
        'reproduksi_berhasil': bool(n_cocok == n_total_cek),
    },
}
simpan_json(hasil_baseline_v2, 'hasil_baseline_v2')

garis('KESIMPULAN VERIFIKASI REPRODUKSI')
print(f'Metrik yang dicek     : {n_total_cek} (3 model x 6 metrik)')
print(f'Cocok (selisih <= {TOLERANSI}) : {n_cocok} / {n_total_cek}')
print(f'Selisih absolut maks  : {selisih_maks:.4f}')
print(f'Selisih absolut rata2 : {selisih_rata:.4f}')
print()
if n_cocok == n_total_cek:
    print('STATUS: REPRODUKSI BERHASIL.')
    print('Pipeline V3 menghasilkan angka yang setara dengan laporan skripsi V2, sehingga')
    print('seluruh eksperimen tambahan pada notebook 01-05 berdiri di atas basis yang sah')
    print('dan hasilnya dapat langsung dibandingkan dengan naskah lama.')
else:
    print('STATUS: TERDAPAT SELISIH DI LUAR TOLERANSI.')
    print('Periksa versi scikit-learn/imbalanced-learn pada runtime, karena perbedaan versi')
    print('dapat menggeser hasil SMOTE dan kalibrasi. Baris berstatus PERIKSA:')
    for _, r in df_verifikasi[df_verifikasi['status'] == 'PERIKSA'].iterrows():
        print(f"  {r['model']:<15s} {r['metrik']:<10s} "
              f"V2={r['nilai_v2']:.4f} V3={r['nilai_v3']:.4f} selisih={r['selisih']:+.4f}")

---
## Bagian D - Penyimpanan Data Bersih (CELL 12)

Agar notebook `01`-`06` tidak perlu mengunduh ulang dataset dari Kaggle (dan agar dipastikan
memakai data yang **persis sama**), data bersih hasil CELL 4 disimpan sebagai berkas CSV.

In [ ]:
# ============================================================
# CELL 12: Simpan Data Bersih untuk Dipakai Ulang Notebook Lain
# ============================================================
garis('SIMPAN DATA BERSIH')

PATH_DATA_BERSIH = f'{OUTPUT_DIR}/data_bersih.csv'
df_clean.to_csv(PATH_DATA_BERSIH, index=False)
ukuran_mb = os.path.getsize(PATH_DATA_BERSIH) / (1024 ** 2)

print(f'[DATA DISIMPAN] {PATH_DATA_BERSIH}')
print(f'Dimensi   : {df_clean.shape[0]:,} baris x {df_clean.shape[1]} kolom')
print(f'Kolom     : {list(df_clean.columns)}')
print(f'Ukuran    : {ukuran_mb:.2f} MB')
print()

# Metadata data bersih (dipakai notebook 06 dan website)
meta_data = {
    'path'              : PATH_DATA_BERSIH,
    'n_baris'           : int(df_clean.shape[0]),
    'n_kolom'           : int(df_clean.shape[1]),
    'fitur'             : SELECTED_FEATURES,
    'label_fitur'       : FEATURE_LABELS,
    'target'            : TARGET,
    'n_kelas_0'         : int((df_clean[TARGET] == 0).sum()),
    'n_kelas_1'         : int((df_clean[TARGET] == 1).sum()),
    'persen_positif'    : round(float(df_clean[TARGET].mean() * 100), 4),
    'praproses'         : ['drop_duplicates', 'seleksi 5 fitur', 'winsorization IQR 1.5'],
    'sumber'            : 'kagglehub: iammustafatz/diabetes-prediction-dataset',
}
simpan_json(meta_data, 'metadata_data_bersih')

print()
print('Cara memakai ulang di notebook 01-06 (OPSIONAL, hanya jika PAKAI_DRIVE = True):')
print('  df_clean = pd.read_csv(f"{OUTPUT_DIR}/data_bersih.csv")')
print('  X_all = df_clean[SELECTED_FEATURES].copy()')
print('  y_all = df_clean[TARGET].copy()')
print()
print('Catatan: bila PAKAI_DRIVE = False, berkas hanya bertahan selama sesi Colab aktif.')
print('Untuk penggunaan lintas notebook, set PAKAI_DRIVE = True pada CELL 2 semua notebook,')
print('atau cukup panggil ulang muat_dan_bersihkan_data() yang hasilnya deterministik sama.')

---
## Bagian E - Ringkasan untuk Skripsi (CELL 13)

Cell berikut mencetak paragraf ringkasan yang dapat **langsung disalin** ke naskah skripsi
(Bab 3 Metodologi dan Bab 4 Hasil), berisi karakteristik data, penanganan ketimpangan kelas,
dan pernyataan konsistensi antar-notebook revisi.

In [ ]:
# ============================================================
# CELL 13: RINGKASAN UNTUK SKRIPSI
# ============================================================
garis('RINGKASAN UNTUK SKRIPSI')

n_total  = len(df_clean)
n_sehat  = int((df_clean[TARGET] == 0).sum())
n_diabet = int((df_clean[TARGET] == 1).sum())
pct_diab = n_diabet / n_total * 100

print('A. KARAKTERISTIK DATA')
print(f'   Dataset yang digunakan adalah Diabetes Prediction Dataset (Kaggle,')
print(f'   iammustafatz/diabetes-prediction-dataset). Setelah penghapusan data duplikat,')
print(f'   diperoleh {n_total:,} baris pengamatan yang unik. Analisis dibatasi pada 5 fitur')
print(f'   terpilih, yaitu usia, BMI, riwayat hipertensi, kadar HbA1c, dan kadar glukosa')
print(f'   darah, dengan variabel target berupa status diabetes (0 = sehat, 1 = diabetes).')
print()

print('B. KETIMPANGAN KELAS')
print(f'   Distribusi kelas sangat timpang: {n_sehat:,} kasus sehat ({100-pct_diab:.2f}%)')
print(f'   berbanding {n_diabet:,} kasus diabetes ({pct_diab:.2f}%). Ketimpangan sekitar')
print(f'   {pct_diab:.1f} persen kelas positif ini membuat akurasi menjadi metrik yang')
print(f'   menyesatkan, sebab model yang memprediksi seluruh sampel sebagai sehat sudah')
print(f'   mencapai akurasi {100-pct_diab:.2f} persen tanpa manfaat klinis apa pun. Karena itu')
print(f'   penelitian ini menetapkan recall, F1-score, dan ROC-AUC sebagai metrik utama,')
print(f'   menerapkan SMOTE hanya pada data latih di dalam pipeline (sehingga tidak terjadi')
print(f'   kebocoran data), serta menggunakan pembagian data dan validasi silang yang')
print(f'   bersifat stratified pada seluruh eksperimen.')
print()

print('C. PRAPROSES')
print(f'   Praproses terdiri atas tiga tahap: penghapusan duplikat, seleksi 5 fitur, dan')
print(f'   winsorization berbasis IQR (batas 1,5 x IQR) pada fitur numerik. Winsorization')
print(f'   dipilih daripada penghapusan baris agar tidak ada pengamatan yang hilang,')
print(f'   terutama pada kelas minoritas. Jumlah nilai yang terkena capping dilaporkan')
print(f'   secara eksplisit pada Tabel eda_outlier.')
print()

print('D. KONSISTENSI ANTAR-EKSPERIMEN')
print(f'   Seluruh notebook revisi (00 sampai 06) menggunakan basis data yang identik')
print(f'   ({n_total:,} baris, 5 fitur), fungsi pemuatan data yang sama, pabrik pipeline yang')
print(f'   sama (StandardScaler -> SMOTE -> classifier dalam imblearn Pipeline), fungsi')
print(f'   evaluasi yang sama, serta random_state = {RANDOM_STATE} yang sama. Konsekuensinya,')
print(f'   setiap perbedaan angka antar-eksperimen murni berasal dari faktor yang sedang')
print(f'   diuji (rasio split, nilai k, kernel/parameter SVM, skema validasi, atau ablasi),')
print(f'   bukan dari perbedaan data maupun praproses. Dengan demikian seluruh perbandingan')
print(f'   antar-eksperimen dalam revisi ini dapat dinyatakan adil dan dapat direplikasi.')
print()

print('E. VERIFIKASI BASELINE')
print(f'   Sebelum menambahkan eksperimen baru, angka baseline V2 direproduksi ulang pada')
print(f'   split 80:20 stratified dengan random_state = {RANDOM_STATE}. Hasil reproduksi')
print(f'   dilaporkan pada Tabel verifikasi_reproduksi_v2 beserta kolom selisih terhadap')
print(f'   angka laporan V2, sehingga pembaca dapat memastikan bahwa temuan-temuan baru')
print(f'   pada notebook 01 sampai 05 berdiri di atas hasil lama yang sudah tervalidasi.')
print()

garis('BERKAS YANG DIHASILKAN NOTEBOOK 00')
print(f'Folder output : {OUTPUT_DIR}')
print('Tabel  : eda_distribusi_kelas, eda_korelasi, eda_distribusi_fitur, eda_outlier,')
print('         verifikasi_reproduksi_v2')
print('Gambar : eda_distribusi_kelas, eda_korelasi, eda_distribusi_fitur, eda_outlier,')
print('         verifikasi_baseline_v2')
print('JSON   : hasil_baseline_v2, metadata_data_bersih')
print('Data   : data_bersih.csv')
print()
print('LANGKAH BERIKUTNYA: jalankan notebook 01 (justifikasi rasio split 80:20),')
print('lalu 02 (pemilihan k KNN), 03 (hyperplane SVM), 04 (validasi statistik),')
print('05 (ablation & robustness), dan terakhir 06 (model final + export produksi).')

---

### Status notebook 00

| Keluaran | Nama berkas | Kegunaan |
|---|---|---|
| Tabel | `eda_distribusi_kelas.csv` | Bukti ketimpangan kelas untuk Bab 4 |
| Tabel | `eda_korelasi.csv` | Matriks korelasi antar variabel |
| Tabel | `eda_distribusi_fitur.csv` | Statistik per kelas + Cohen d tiap fitur |
| Tabel | `eda_outlier.csv` | Jumlah outlier IQR sebelum/sesudah winsorization |
| Tabel | `verifikasi_reproduksi_v2.csv` | Perbandingan angka V2 vs reproduksi V3 + selisih |
| Gambar | `eda_distribusi_kelas.png`, `eda_korelasi.png`, `eda_distribusi_fitur.png`, `eda_outlier.png`, `verifikasi_baseline_v2.png` | Lampiran gambar skripsi |
| JSON | `hasil_baseline_v2.json`, `metadata_data_bersih.json` | Dibaca notebook `06` dan website |
| Data | `data_bersih.csv` | Basis data identik untuk notebook `01`-`06` |

Fondasi siap. Lanjutkan ke `01_Justifikasi_Rasio_Split.ipynb`.